# GenAI in Healthcare— Assist, Don’t Replace

**Module 16 · Lesson 16.1**  
*Netsetos GenAI Engineering Course v2.0*

**In this lesson you will:**
- The Golden Rule: Assist, Don’t Replace
- The SOAP Note Validator
- The Medical Q&A Guardrail Gate
- The Radiology Structured Report
- PHI De-Identification (HIPAA Safe Harbor)
- The Grounding / Hallucination Check

---

> This notebook is generated from the published lesson HTML. The HTML is the source of truth - do not hand-edit this notebook. Re-run the generator if the lesson updates.


In [ ]:
# Reproducibility - the lesson uses seeded random values throughout

## The note nobody should have signed

> 💡 **Info**
>
> A hospital pilots an AI scribe that drafts the visit note from the conversation. For a month it is a quiet miracle — hours of paperwork handed back to clinicians every day. Then one note records a medication the patient was never prescribed: a fluent, confident, entirely fabricated line, sitting in the Plan as if a doctor had written it. A rushed clinician almost signs it. Nobody was reckless; the model simply did what models do, and there was no machinery to catch it. That single scene is the whole lesson. In medicine the model is a **co-pilot** and the clinician is always the **pilot**, and the engineering job is not to make the model smarter — it is to build the guardrails that keep the human in command. Over the next seven steps you will build those guardrails as runnable checks: a classifier that decides how much oversight a use case needs, a validator that catches a note missing its exam data, a gate that refuses individual dosing and escalates an emergency, a schema that pages a radiologist on a critical finding, a de-identifier that strips PHI before the model ever sees it, a grounding check that blocks a hallucinated drug, and a release gate that will not ship an output nobody signed.

> 📦 **Info**
>
> ⚠️ Read this first Everything here is an **educational engineering prototype**, not a clinical tool. The examples are illustrative and deterministic; nothing in this lesson is medical advice or fit for real patient care. The golden rule is stated once and enforced everywhere: a medical AI **assists** a licensed clinician and never replaces one, all clinical outputs require human review and sign-off, and a chatbot never diagnoses, prescribes, or handles an emergency on its own.

### 🎯 What you will be able to do after this lesson

- **Apply** a SOAP-note structure validator and a radiology report schema to check that an AI-drafted clinical document is well-formed before a clinician reads it.

- **Analyze** a medical AI use case for the human oversight it requires, and a Q&A query for whether it must be answered, refused, or escalated.

- **Evaluate** an AI clinical summary against its source for unsupported claims, and a record for the PHI that must be removed before a model call.

- **Create** a medical AI release gate that ties de-identification, grounding, guardrails, clinician sign-off, and audit logging into one ship / no-ship decision.

> 📦 **Info**
>
> ✅ Before you startThis is the **opener** of Module 16, which tours the industries where GenAI ships — and healthcare is where the safety discipline is turned up to its highest setting. It stands on work you have already done: the PII scrubbing from **Lesson 15.4** (here it becomes PHI de-identification), the structured-output schemas from **Lesson 5.5** (here they validate a SOAP note and a radiology report), the grounding and hallucination ideas from **Module 15** and **Module 08**, and the API engineering from **Module 07** (the model call is shown illustratively). Everything runnable here is keyless rule-checking on an illustrative clinical example.

## 60-Second Warm-Up

Flip each card and answer before revealing.

> 🩹 **Analogy**
>
> A medical AI is a **co-pilot, never the pilot**. A co-pilot is genuinely useful — it reads the instruments, drafts the flight plan, flags what the captain might have missed, and carries real load on a long flight. But it does not decide to land, and it never overrides the captain; command authority stays with one accountable human the whole way. A clinical model is exactly that co-pilot: it drafts the note, surfaces the relevant history, formats the report — and the clinician holds command, makes the call, and signs. The instant the assistant is allowed to make the final clinical decision, you have handed the controls to the co-pilot, which is the one thing the role forbids. **Where it breaks down:** a real co-pilot is a licensed pilot who *can* legally fly the plane and sometimes does under supervision, whereas a clinical model has no license and never makes the final call at all — so weight the analogy toward “the human always holds command authority,” not “the assistant occasionally takes the controls.”

> 📦 **Info**
>
> 🚫 Three misconceptions this lesson kills
> - **“If the model is accurate enough, it can decide.”** No — anything that touches a clinical decision needs a clinician in the loop; accuracy never buys out the human.
> - **“A disclaimer makes a chatbot safe to give medical advice.”** A disclaimer does not license individual diagnosis or dosing; the gate must still refuse and redirect.
> - **“De-identification is a privacy nicety.”** It is a legal boundary — a model call or a log line carrying raw PHI is a reportable breach you cannot undo.

> 💡 **Info**
>
> ⚠️ Anti-patternThe demo-grade medical AI: it generates a beautiful SOAP note or radiology report on stage, and behind it there is no structure validation, no grounding check against the source, no PHI scrub, and no sign-off. It looks production-ready in a slide deck and is a patient-safety incident waiting to happen. A medical AI output that nobody validated, grounded, de-identified, or signed is not a feature — it is a liability with good formatting. This whole lesson is the opposite: the model’s fluent draft is the *start* of the pipeline, and every guardrail after it is what makes the draft safe to touch a patient’s chart.

---

## 🎯 Concept 1: The Golden Rule: Assist, Don’t Replace

### The Golden Rule: Assist, Don’t Replace

Before any code, classify the use case. Anything that touches a clinical decision needs a clinician in the loop; fully autonomous diagnosis or treatment is off the table. The control scales with autonomy - from a “not medical advice” disclaimer, to mandatory clinician sign-off, to prohibited - and that decision governs everything you build after it.

Start with the decision that comes before any model call: *how much human control does this use case require?* The answer does not depend on how good the model is — it depends on what the use case *touches*. A use case that only provides general health information (“what is a normal resting heart rate?”) is **informational**: answer it, but attach a “not medical advice” disclaimer, because no clinical decision is being made. A use case that drafts a clinical note or suggests a triage level *does* touch a clinical decision, so it is **assistive**: the model drafts, and a clinician reviews and signs — human-in-the-loop is mandatory, not optional. And a use case that would *autonomously* diagnose and prescribe with no human sign-off is **prohibited**, no matter how accurate it tests, because a fully autonomous clinical decision is not an accepted deployment. In the worked example, four use cases sort into one informational, two assistive, and one prohibited. This classifier is the spine of the whole lesson: it is the golden rule — *assist, don’t replace* — expressed as a rule you can run, and it decides which guardrails every later step has to enforce. The block classifies use cases by their required control, keyless.

> ✍️ **Analogy**
>
> Think of a **courtroom scribe, not the judge**. The scribe captures every word of the proceeding, drafts the record, and hands it up — indispensable, and yet the scribe never rules on the case. The ruling is the judge’s alone, signed by the judge, carrying the judge’s accountability. A clinical AI is the scribe: it can draft the note and organize the facts, but the diagnosis and the plan are the clinician’s ruling, made and signed by the clinician. The moment the scribe starts issuing verdicts, the whole system has failed — not because the scribe is unskilled, but because deciding was never the scribe’s role. Sort every use case by that line: is the model drafting the record, or issuing the verdict?

An AI symptom chatbot tests as highly accurate. Can it diagnose and prescribe on its own?

**📝 `01_assist_dont_replace.py`** - *runnable*

In [ ]:
# THE GOLDEN RULE - ASSIST, DON'T REPLACE: in medicine a model is a co-pilot, never the pilot. Classify each use case by whether
# it touches a clinical decision and how autonomous it is; the control scales from a disclaimer to mandatory clinician sign-off.
def control(touches_clinical, autonomous):
    if touches_clinical and autonomous:
        return "PROHIBITED - a licensed clinician must make or approve every diagnosis and treatment"
    if touches_clinical:
        return "ASSISTIVE - a clinician reviews and signs off (human-in-the-loop is mandatory)"
    return "INFORMATIONAL - add a 'not medical advice' disclaimer; no clinical decision is made"
USE_CASES = [  # (use case, touches a clinical decision?, fully autonomous?)
    ("general symptom-information chatbot", False, False),
    ("draft a clinical note from a visit",  True,  False),
    ("suggest a triage acuity level",       True,  False),
    ("autonomously diagnose and prescribe", True,  True)]
print("Classifying medical AI use cases by the required human control:")
counts = {}
for name, clinical, auto in USE_CASES:
    c = control(clinical, auto)
    tier = c.split(" -")[0]
    counts[tier] = counts.get(tier, 0) + 1
    print("  {:<36} -> {}".format(name, c))
print()
print("Tally: {}.".format(", ".join("{} {}".format(v, k) for k, v in counts.items())))
print("The rule is not a slogan: anything that touches a clinical decision needs a clinician in the loop, and fully autonomous")
print("diagnosis or treatment is off the table. The engineering job is to build the guardrails that keep the human in command.")

# Output:
# Classifying medical AI use cases by the required human control:
#   general symptom-information chatbot  -> INFORMATIONAL - add a 'not medical advice' disclaimer; no clinical decision is made
#   draft a clinical note from a visit   -> ASSISTIVE - a clinician reviews and signs off (human-in-the-loop is mandatory)
#   suggest a triage acuity level        -> ASSISTIVE - a clinician reviews and signs off (human-in-the-loop is mandatory)
#   autonomously diagnose and prescribe  -> PROHIBITED - a licensed clinician must make or approve every diagnosis and treatment
#
# Tally: 1 INFORMATIONAL, 2 ASSISTIVE, 1 PROHIBITED.
# The rule is not a slogan: anything that touches a clinical decision needs a clinician in the loop, and fully autonomous
# diagnosis or treatment is off the table. The engineering job is to build the guardrails that keep the human in command.

- Each use case is classified by two things: does it touch a clinical decision, and is it fully autonomous.
- A general-information chatbot is **informational** (disclaimer); drafting a note or suggesting triage is **assistive** (clinician signs off).
- A use case that would autonomously diagnose and prescribe is **PROHIBITED** — accuracy does not change that.
- The tally — 1 informational, 2 assistive, 1 prohibited — is the golden rule as a runnable rule: assist, don’t replace.

#### 💡 What just happened

⚡ What just happened? The oversight classifier turns “assist, don’t replace” into a rule: informational use cases get a disclaimer, assistive ones get mandatory clinician sign-off, and fully autonomous clinical decisions are prohibited. Tradeoff: none worth having - a model that is not allowed to decide is the point, not a limitation. Next: for the assistive case, validate the structure of what the model drafts - the SOAP note.

- Four use cases sorting into informational / assistive / prohibited
- 1 informational, 2 assistive, 1 prohibited

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 2: The SOAP Note Validator

### The SOAP Note Validator

An AI-drafted clinical note follows the SOAP structure - Subjective, Objective, Assessment, Plan - and must not invent an Assessment or Plan with no Objective data behind it. Validate the structure before a clinician reads it: catch the missing section and the fabrication pattern first, so the clinician spends their review on judgement, not on the machine’s clerical mistakes.

The most common assistive use case is documentation, and clinical notes have a standard shape: **SOAP** — *Subjective* (what the patient reports), *Objective* (the exam findings, vitals, labs), *Assessment* (the clinical impression), and *Plan* (next steps). A well-formed AI draft carries all four. But structure validation is not just a box-ticking check for completeness; it also catches a specific and dangerous failure. If a note has an *Assessment* and a *Plan* but no *Objective* data, that is a **fabrication red flag** — a conclusion with nothing observed to support it is exactly the pattern a hallucinating model produces, confidently stating an impression it has no findings for. In the worked example the validator confirms all four sections are present and the note is valid, then shows what happens when Objective is removed: the fabrication guard fires and the note is blocked for review. The clinician still reads and signs every note — the validator does not replace that judgement, it protects it, by making sure the human spends their scarce attention on the medicine rather than on catching the machine’s structural mistakes. The block validates the SOAP structure and the fabrication guard, keyless.

> 📋 **Analogy**
>
> The SOAP structure is a **structured incident report, where every field is required for a reason**. A good incident form will not let you write the “root cause” and “corrective action” while leaving “what was observed” blank — because a conclusion with no observations behind it is not an analysis, it is a guess wearing a form’s authority. Insisting the Objective section be filled before an Assessment is trusted is the same safeguard: it forces the reasoning to rest on something observed, and it makes a conclusion-without-evidence visible as the empty field it depends on. The form is not bureaucracy; it is a shape that makes a missing foundation impossible to hide.

An AI note has a filled-in Assessment and Plan but an empty Objective section. Ready for the chart?

**📝 `02_soap_validator.py`** - *runnable*

In [ ]:
# THE SOAP NOTE VALIDATOR: an AI-drafted clinical note must follow the SOAP structure - Subjective, Objective, Assessment, Plan -
# and it must not invent an Assessment or Plan without Objective data to support it. Validate the STRUCTURE before a clinician reads it.
REQUIRED = ["subjective", "objective", "assessment", "plan"]
def validate_soap(note):
    present = [s for s in REQUIRED if s in note and note[s].strip()]
    missing = [s for s in REQUIRED if s not in present]
    # a fabrication guard: an Assessment/Plan with no Objective findings is a red flag
    unsupported = bool(note.get("assessment") or note.get("plan")) and not note.get("objective", "").strip()
    return present, missing, unsupported
note = {"subjective": "Patient reports a sore throat for 3 days.",
        "objective": "Temp 37.9C, pharyngeal erythema, no exudate.",
        "assessment": "Likely viral pharyngitis.",
        "plan": "Supportive care; return if worse."}
present, missing, unsupported = validate_soap(note)
print("SOAP structure check ({} required sections):".format(len(REQUIRED)))
for s in REQUIRED:
    print("  [{}] {}".format("x" if s in present else " ", s.capitalize()))
print()
ok = not missing and not unsupported
print("verdict: {}".format("VALID - ready for clinician review" if ok else "INVALID"))
if missing: print("   - missing section(s): {}".format(", ".join(missing)))
if unsupported: print("   - Assessment/Plan present with no Objective data - possible fabrication, block for review")
print("A clinician still reads and signs every note. The validator just catches the machine's structural mistakes first.")

# Output:
# SOAP structure check (4 required sections):
#   [x] Subjective
#   [x] Objective
#   [x] Assessment
#   [x] Plan
#
# verdict: VALID - ready for clinician review
# A clinician still reads and signs every note. The validator just catches the machine's structural mistakes first.

- The validator checks all four SOAP sections are present and non-empty: Subjective, Objective, Assessment, Plan.
- All four present → **VALID**, ready for clinician review.
- The fabrication guard: an Assessment or Plan with *no* Objective data is a red flag — a conclusion with no findings behind it.
- A clinician still signs every note; the validator just catches the machine’s structural mistakes first, so the human reviews the medicine.

#### 💡 What just happened

⚡ What just happened? The SOAP validator confirms a note has all four sections and fires a fabrication guard when an Assessment or Plan has no Objective data behind it - the pattern a hallucinating model produces. Tradeoff: structure validation catches clerical and fabrication defects, not clinical correctness - the clinician still owns that. Next: the patient-facing case, where the guardrail gate runs before the model answers at all.

- A four-section note validating: Subjective, Objective, Assessment, Plan
- Remove Objective → the fabrication guard fires

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 3: The Medical Q&A Guardrail Gate

### The Medical Q&A Guardrail Gate

A patient-facing model must never give individual diagnosis or dosing, must escalate an emergency, and must attach a disclaimer and a source. The gate runs BEFORE the model answers: emergency signs route to “call emergency services now,” individual-advice requests route to refuse-and-redirect, and only general-education questions are answered - with a citation and a disclaimer.

Now the harder case: a model talking directly to a patient. The safety cannot live in the answer, because by the time the model has answered, the unsafe advice already exists — so the **guardrail gate runs before the model responds**. It makes three decisions. If the question contains *emergency signs* (chest pain, trouble breathing, suicidal ideation, stroke, severe bleeding), it does not try to help — it **escalates**, telling the user to call emergency services now, because an AI response is the wrong thing to be doing in those seconds. If the question asks for *individualized advice* — a specific dose, a personal diagnosis (“what should I take,” “do I have X”) — it **refuses and redirects** to a clinician or pharmacist, because individual dosing and diagnosis are exactly what a chatbot must not give, over-the-counter or not. Everything else — general health education — it **answers**, with a citation and a “not medical advice” disclaimer. In the worked example, three questions route cleanly to answer, refuse-and-redirect, and escalate. The gate is deliberately conservative: general education is fine, but individualized advice and emergencies are routed away from the model, every time, with no exceptions. The block runs the guardrail gate, keyless.

> 💊 **Analogy**
>
> The gate is a **pharmacist who will not fill a dangerous prescription**. A good pharmacist answers general questions all day — what a drug is for, what the common side effects are — but the instant a request crosses into “just give me a high dose of this without a prescription,” they stop and redirect you to a doctor, and if you walk in describing a heart attack, they do not reach for a shelf — they call an ambulance. The refusal is not unhelpfulness; it is the most helpful and safest thing they can do, because some requests are outside what anyone at that counter should handle. The guardrail gate is that judgement encoded: answer the general question, refuse the dangerous individual one, and escalate the emergency — before the model ever opens its mouth.

A user asks: “What dose of ibuprofen should I take for my back?” What should the gate do?

**📝 `03_guardrail_gate.py`** - *runnable*

In [ ]:
# THE MEDICAL Q&A GUARDRAIL GATE: a patient-facing model must never give individualized diagnosis or dosing, must escalate an
# emergency, must add a disclaimer, and must cite a source. Run every question through the gate before the answer goes out.
EMERGENCY = {"chest pain", "trouble breathing", "suicidal", "stroke", "severe bleeding"}
INDIVIDUAL_ADVICE = {"what dose", "should i take", "do i have", "diagnose me", "how much of"}
def guardrail(question):
    q = question.lower()
    if any(e in q for e in EMERGENCY):
        return "ESCALATE - emergency signs: tell the user to call emergency services now"
    if any(a in q for a in INDIVIDUAL_ADVICE):
        return "REFUSE + REDIRECT - no individual diagnosis/dosing; advise seeing a clinician"
    return "ANSWER - general info only, with a citation and a 'not medical advice' disclaimer"
QUESTIONS = [
    "What is a normal resting heart rate?",
    "What dose of ibuprofen should I take for my back?",
    "I am having chest pain and trouble breathing, what do I do?"]
print("Guardrail decisions for three patient questions:")
for q in QUESTIONS:
    print("  {:<52} -> {}".format('"' + q[:48] + ('..."' if len(q) > 48 else '"'), guardrail(q).split(" -")[0]))
print()
for q in QUESTIONS:
    print("- {}\n    {}".format(q, guardrail(q)))
print("The gate runs BEFORE the model answers. General education is fine; individualized medical advice and emergencies are not")
print("things a chatbot handles - it redirects to a clinician or to emergency services, every time, with no exceptions.")

# Output:
# Guardrail decisions for three patient questions:
#   "What is a normal resting heart rate?"               -> ANSWER
#   "What dose of ibuprofen should I take for my back..." -> REFUSE + REDIRECT
#   "I am having chest pain and trouble breathing, wh..." -> ESCALATE
#
# - What is a normal resting heart rate?
#     ANSWER - general info only, with a citation and a 'not medical advice' disclaimer
# - What dose of ibuprofen should I take for my back?
#     REFUSE + REDIRECT - no individual diagnosis/dosing; advise seeing a clinician
# - I am having chest pain and trouble breathing, what do I do?
#     ESCALATE - emergency signs: tell the user to call emergency services now
# The gate runs BEFORE the model answers. General education is fine; individualized medical advice and emergencies are not
# things a chatbot handles - it redirects to a clinician or to emergency services, every time, with no exceptions.

- The gate runs **before** the model answers and makes one of three decisions.
- Emergency signs (chest pain, trouble breathing) → **ESCALATE**: call emergency services now.
- A request for individual dosing or diagnosis → **REFUSE + REDIRECT** to a clinician or pharmacist.
- A general-education question → **ANSWER**, with a citation and a “not medical advice” disclaimer — every time, no exceptions.

#### 💡 What just happened

⚡ What just happened? The guardrail gate runs before the model answers and routes each question to ANSWER (general education + disclaimer), REFUSE + REDIRECT (individual dosing/diagnosis), or ESCALATE (emergency signs). Tradeoff: a conservative gate will sometimes refuse a question it could have safely answered - the right error to make in medicine. Next: another structured-output case with a safety twist - the radiology report and its critical flag.

- Three patient questions routing through the gate
- ANSWER / REFUSE + REDIRECT / ESCALATE

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 4: The Radiology Structured Report

### The Radiology Structured Report

Free-text findings become a structured report with required fields - modality, findings, impression, recommendation, and a critical flag - and any critical finding is routed to an urgent read. Structure is not cosmetic here: the critical field is what pages a radiologist in minutes instead of hours, so a critical finding detected but not flagged is a safety bug, not a formatting slip.

Radiology gives structured output a life-or-death edge. A report has required fields — *modality*, *findings*, *impression*, *recommendation*, and a *critical* flag — and validating that schema is the baseline. But the field that matters most is the critical flag, because it is not documentation, it is *routing*. When the findings contain a critical term — a tension pneumothorax, a hemorrhage, an aortic dissection — the report must set its critical flag, and that flag is what pages a radiologist for an **urgent read**, turning a hours-long queue into a minutes-long one. So the validator checks two things: that all the fields are present, and that the critical flag *matches* what the findings actually say. A report that is perfectly well-formed but has a critical finding with the flag left off is not a formatting slip — it is a routing bug that could let a life-threatening finding sit unread in a normal queue. In the worked example the report is complete, the pneumothorax is detected, the critical flag is set, and the report routes to an urgent read; the block also shows how a mismatch (detected but not flagged) is caught. The block validates the schema and the critical routing, keyless.

> 🏷️ **Analogy**
>
> The critical flag is a **triage tag on a patient arriving at the emergency department**. Two patients can have complete, identical paperwork, but the red tag versus the green tag is what decides who is seen in the next sixty seconds and who waits — the tag is not a description, it is a routing decision with a clock attached. A radiology report’s critical flag works the same way: a beautifully formatted report with the flag left off a genuine emergency is a red-tag patient wearing a green tag, sent to the waiting room. Checking that the flag matches the findings is checking that the triage tag matches the patient — the single most consequential field on the form.

A report’s findings mention a tension pneumothorax, but its critical flag is off. Ready to queue?

**📝 `04_radiology_report.py`** - *runnable*

In [ ]:
# THE RADIOLOGY STRUCTURED-REPORT SCHEMA: free-text findings become a structured report with required fields, and any CRITICAL
# finding is flagged for an urgent read. Validate the schema and the critical-flag routing before the report reaches the queue.
SCHEMA = ["modality", "findings", "impression", "recommendation", "critical"]
CRITICAL_TERMS = {"pneumothorax", "hemorrhage", "aortic dissection", "free air", "mass effect"}
def validate_report(r):
    missing = [f for f in SCHEMA if f not in r]
    text = (r.get("findings", "") + " " + r.get("impression", "")).lower()
    should_flag = any(t in text for t in CRITICAL_TERMS)
    flag_ok = r.get("critical", False) == should_flag
    return missing, should_flag, flag_ok
report = {"modality": "chest CT",
          "findings": "Large right-sided pneumothorax with mediastinal shift.",
          "impression": "Tension pneumothorax.",
          "recommendation": "Immediate clinical correlation and decompression.",
          "critical": True}
missing, should_flag, flag_ok = validate_report(report)
print("Radiology report schema check ({} required fields):".format(len(SCHEMA)))
for f in SCHEMA:
    print("  [{}] {}".format("x" if f in report else " ", f))
print()
print("critical finding detected: {}  |  report flagged critical: {}  |  routing {}".format(
    should_flag, report.get("critical", False), "OK" if flag_ok else "MISMATCH - fix before queueing"))
print("verdict: {}".format("VALID - route to urgent read" if not missing and flag_ok and should_flag else "check the report"))
print("Structured output is not just tidy - the 'critical' field is what pages a radiologist in minutes instead of hours.")

# Output:
# Radiology report schema check (5 required fields):
#   [x] modality
#   [x] findings
#   [x] impression
#   [x] recommendation
#   [x] critical
#
# critical finding detected: True  |  report flagged critical: True  |  routing OK
# verdict: VALID - route to urgent read
# Structured output is not just tidy - the 'critical' field is what pages a radiologist in minutes instead of hours.

- The schema requires five fields: modality, findings, impression, recommendation, and a critical flag.
- The validator checks completeness *and* that the critical flag matches the findings text.
- A critical term (tension pneumothorax) is detected, the flag is set, and the report routes to an **urgent read**.
- Detected-but-not-flagged is a routing MISMATCH — a safety bug, because the flag is what pages the radiologist urgently.

#### 💡 What just happened

⚡ What just happened? The radiology schema validates five required fields and checks that the critical flag matches the findings - the flag is routing, not documentation, and a critical finding not flagged never triggers the urgent read. Tradeoff: keyword-based critical detection is a coarse safety net, not a substitute for the radiologist’s read - it errs toward flagging. Next: the boundary every one of these systems shares - stripping PHI before the model call.

- A five-field report filling in with a critical flag
- Pneumothorax → critical → urgent read

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 5: PHI De-Identification (HIPAA Safe Harbor)

### PHI De-Identification (HIPAA Safe Harbor)

Before any clinical text reaches a model or a log, strip the protected health information. HIPAA’s Safe Harbor method enumerates 18 identifier categories - names, dates, record numbers, contacts. You cannot honour privacy after a leak, so de-identify at ingestion: a model call or a log line that carries raw PHI is a reportable breach you cannot take back. This applies Lesson 15.4’s PII work to the clinical setting.

Every system in this lesson shares one boundary: the moment clinical text leaves your control to reach a hosted model — or gets written to an application log. Before it crosses that boundary, the **protected health information** must be stripped. HIPAA’s *Safe Harbor* method names 18 identifier categories to remove: names, geographic detail finer than a state, every date element tied to an individual, phone and fax numbers, email, Social Security numbers, medical record and account numbers, biometric identifiers, full-face photographs, and more. The rule is **de-identify at ingestion, not after**, and the reason is unforgiving: once raw PHI has been sent in a model call or written to a log, the breach has already happened — you cannot un-send it, and in a covered setting it is reportable. This is Lesson 15.4’s PII scrubbing carried into the clinical world, where the identifiers have a legal definition and the penalties are real. In the worked example, a patient record with seven identifier fields is scrubbed to tokens (“[NAME],” “[MRN],” and so on) while the clinical note — which carries no PHI — is retained, and the readout confirms seven identifiers removed. The clinical signal survives; the identifiers do not. The block de-identifies a record at the boundary, keyless.

> 📧️ **Analogy**
>
> De-identification is **redacting a document before it leaves the building**. When a sensitive file is released under a public-records request, someone runs a black marker over the names, addresses, and account numbers *before* it goes out the door — because once a copy with the names on it has been handed over, no amount of regret pulls it back. You do not release the original and hope the recipient is discreet; you redact at the boundary, every time, and only the redacted copy travels. Sending clinical text to a model is that release: scrub the identifiers before the text crosses the line, because the boundary is the last place you still control what leaves. **Where it breaks down:** a black marker only hides what you thought to cover, and real PHI hides in free text and rare formats too, so Safe Harbor field-stripping is a floor, not a proof of anonymity.

You want to send a clinical note to a hosted model. When do you remove the PHI?

**📝 `05_phi_deidentification.py`** - *runnable*

In [ ]:
# PHI DE-IDENTIFICATION (HIPAA Safe Harbor): before any text goes to a model or a log, strip the protected health information -
# names, record numbers, dates, contacts. You cannot honour privacy after a leak; scrub at the boundary. (Ties to Lesson 15.4.)
PHI_FIELDS = {"name": "[NAME]", "mrn": "[MRN]", "dob": "[DATE]", "phone": "[PHONE]",
              "address": "[ADDRESS]", "email": "[EMAIL]", "ssn": "[SSN]"}
record = {"name": "Jane Doe", "mrn": "MRN-4471902", "dob": "1984-02-11", "phone": "555-0142",
          "address": "12 Oak St", "email": "jane@x.com", "ssn": "123-45-6789",
          "note": "Patient with sore throat, no fever. Follow up in one week."}
scrubbed = dict(record)
removed = []
for field, token in PHI_FIELDS.items():
    if field in scrubbed and scrubbed[field]:
        scrubbed[field] = token
        removed.append(field)
print("PHI de-identification (HIPAA Safe Harbor - 18 identifier categories):")
print("  identifiers removed: {} -> {}".format(len(removed), ", ".join(removed)))
print("  clinical note (retained, no PHI): {}".format(scrubbed["note"]))
print()
print("{} of the record's identifier fields were replaced with tokens before the note left the boundary.".format(len(removed)))
print("De-identify at ingestion, not after: a model call or a log line that carries raw PHI is a breach you cannot take back.")

# Output:
# PHI de-identification (HIPAA Safe Harbor - 18 identifier categories):
#   identifiers removed: 7 -> name, mrn, dob, phone, address, email, ssn
#   clinical note (retained, no PHI): Patient with sore throat, no fever. Follow up in one week.
#
# 7 of the record's identifier fields were replaced with tokens before the note left the boundary.
# De-identify at ingestion, not after: a model call or a log line that carries raw PHI is a breach you cannot take back.

- HIPAA Safe Harbor names 18 identifier categories; the record here carries seven of them (name, MRN, DOB, phone, address, email, SSN).
- Each identifier is replaced with a token (“[NAME],” “[MRN]”) before the text crosses the boundary.
- The clinical note — which carries no PHI — is retained, so the medical signal survives de-identification.
- Seven identifiers removed: de-identify at ingestion, because a model call or log carrying raw PHI is a breach you cannot undo.

#### 💡 What just happened

⚡ What just happened? PHI de-identification strips the HIPAA Safe Harbor identifiers at the boundary - seven removed here - before any text reaches a model or a log, because a leak cannot be undone. Tradeoff: Safe Harbor field-stripping is a floor; real de-identification also has to catch PHI hiding in free text. Next: even de-identified and well-formed, an output can still be wrong - the grounding check.

- A patient record with seven PHI fields scrubbed to tokens
- 7 identifiers removed at the boundary

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 6: The Grounding / Hallucination Check

### The Grounding / Hallucination Check

Every clinical claim in an AI summary must be supported by the source note. A fluent summary is not a correct one: the failure that matters is a confident, well-formatted, fabricated claim - a medication or finding the source never mentioned. Check each claim against the source and block any that is unsupported, because a hallucinated drug that reaches a chart can drive a real treatment decision.

A de-identified, well-structured output can still be *false*, and this is the failure that most defines clinical AI risk. When a model summarizes a note, every clinical claim in that summary must trace back to something the **source** actually said. The dangerous failure mode is not a garbled sentence you would catch at a glance — it is a fluent, confident, perfectly formatted claim about a medication or a finding that the source note never contained. A summary that reads beautifully and states the patient was “prescribed amoxicillin” when no such thing appears in the record is worse than an obviously broken one, because it is exactly the kind of thing a rushed reviewer trusts. So the grounding check does the unglamorous work: it takes each claim, looks for its support in the source, and **blocks any claim that is unsupported**. In the worked example, two claims are grounded and one — the fabricated medication — is flagged as unsupported and the summary is blocked for review. The lesson beneath it: *a fluent summary is not a correct one*, and in medicine the gap between fluent and correct is where a hallucinated drug reaches a chart and drives a real decision. The block grounds each claim against the source, keyless.

> 🔍 **Analogy**
>
> The grounding check is a **fact-checker who verifies every claim against the source**, not against how good it sounds. A rigorous fact-checker does not pass a sentence because it is well written and plausible; they put a finger on the source document and ask, “where exactly does it say this?” — and if the source is silent, the claim is cut, however confident and quotable it was. That is precisely the discipline a clinical summary needs: not “does this read like a real note?” but “is every claim here actually in the record?” The most dangerous sentence is the fluent, specific, unsupported one, because fluency is what makes a fabrication trusted. Grounding replaces trust-by-fluency with trust-by-source.

An AI summary reads fluently and names a medication the source note never mentioned. Trust it?

**📝 `06_grounding_check.py`** - *runnable*

In [ ]:
# THE GROUNDING / HALLUCINATION CHECK: every clinical claim in an AI summary must be supported by the source note. Check each
# claim's key terms against the source; an unsupported claim (a drug the note never mentions) is a hallucination - block it.
source = ("Patient reports sore throat for three days. Exam: temp 37.9C, pharyngeal erythema, no exudate. "
          "Assessment: viral pharyngitis. Plan: supportive care, fluids, rest.")
claims = ["The patient has a sore throat.",
          "The exam showed pharyngeal erythema.",
          "The patient was prescribed amoxicillin."]      # the last claim is NOT in the source
def supported(claim, src):
    stop = {"the", "a", "an", "was", "has", "is", "of", "for", "patient"}
    key = [w.strip(".,").lower() for w in claim.split() if w.strip(".,").lower() not in stop]
    hits = sum(1 for w in key if w in src.lower())
    return hits >= max(1, len(key) - 1)                   # nearly all key terms must appear in the source
print("Grounding check - is each claim supported by the source note?")
unsupported = []
for c in claims:
    ok = supported(c, source)
    if not ok: unsupported.append(c)
    print("  [{}] {}".format("grounded" if ok else "UNSUPPORTED", c))
print()
print("{} unsupported claim(s) -> block the summary and flag for review.".format(len(unsupported)))
for c in unsupported:
    print("   - '{}' - no support in the source (a fabricated medication is a patient-safety event)".format(c))
print("A fluent summary is not a correct one. Ground every clinical claim to the source, or a hallucinated drug reaches a chart.")

# Output:
# Grounding check - is each claim supported by the source note?
#   [grounded] The patient has a sore throat.
#   [grounded] The exam showed pharyngeal erythema.
#   [UNSUPPORTED] The patient was prescribed amoxicillin.
#
# 1 unsupported claim(s) -> block the summary and flag for review.
#    - 'The patient was prescribed amoxicillin.' - no support in the source (a fabricated medication is a patient-safety event)
# A fluent summary is not a correct one. Ground every clinical claim to the source, or a hallucinated drug reaches a chart.

- Each claim in the summary is checked for support in the **source** note, not for how fluent it sounds.
- Two claims (sore throat, pharyngeal erythema) are grounded; one (prescribed amoxicillin) is **UNSUPPORTED** — the note never mentions it.
- The one unsupported claim blocks the summary and flags it for review.
- A fluent summary is not a correct one; a fabricated medication that reaches a chart can drive a real treatment decision.

#### 💡 What just happened

⚡ What just happened? The grounding check verifies every clinical claim against the source note and blocks any that is unsupported - one fabricated medication caught here - because fluency is not correctness. Tradeoff: token-overlap grounding is a coarse first pass; production systems use stronger claim-to-span matching. Next: the check that ties the whole lesson together - the audit trail and the release gate.

- Three claims checked against a source note
- 1 unsupported (amoxicillin) → block

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 7: The Audit Trail + Release Gate

### The Audit Trail + Release Gate

Every medical AI output is logged - model version, input and output hashes, timestamp, clinician sign-off - and gated before release. The gate ties the lesson together: de-identified, grounded, guardrailed, clinician-signed, and audit-logged. One unmet item is a BLOCK. The audit trail is not bureaucracy: when an output is questioned, you must show exactly what model produced what, from what input, and who signed it.

Everything in this lesson converges on one moment: the decision to release a medical AI output. The **release gate** is the checklist that decision has to pass, and it is where the individual guardrails become one accountable yes-or-no. The gate confirms the output was *de-identified* before the model saw it (Step 5), *grounded* to its source (Step 6), passed the *safety guardrails* (Step 3), was *reviewed and signed by a clinician* (the golden rule, Step 1), and was *written to the audit trail*. That audit record — model version, input and output hashes, timestamp, and the clinician sign-off — is not paperwork for its own sake: it is what lets you answer, when an output is later questioned, exactly what model produced what, from what input, and who signed it. The defining property of a real gate is that it can say no: in the worked example, four checks pass but the audit trail is not wired up, so the decision is **BLOCK** — the output does not ship until logging exists, because in a clinical setting an output you cannot trace is an output you cannot defend. This is the discipline the whole module points at: de-identify, ground, guardrail, sign, and log — or it does not ship. The block evaluates the release gate, keyless.

> ✅ **Analogy**
>
> The audit trail and gate together are a **flight recorder and a pre-flight checklist**. The checklist runs before every departure — controls, fuel, sign-off — and a single failed item cancels the flight, no matter the schedule; that is the gate, and its whole value is that it can ground a release. The flight recorder runs the whole time, capturing what happened so that if anything is ever questioned, there is an exact, tamper-evident record instead of a shrug; that is the audit trail. Neither is bureaucracy: the checklist is what stops an unsafe departure, and the recorder is what makes every departure accountable after the fact. A medical AI output ships the same way — it passes every gate check, and it leaves a recording of what produced it and who signed — or it does not leave the ground.

Four of five release-gate checks pass, but the audit trail is not wired up. Do you release the output?

**📝 `07_release_gate.py`** - *runnable*

In [ ]:
# THE AUDIT TRAIL + DEPLOYMENT GATE: every medical AI output is logged (model version, timestamp, a content hash, the clinician
# sign-off) and gated before release. The gate ties the whole lesson together: de-identified, grounded, guardrailed, signed, logged.
def content_hash(text):                                   # a toy deterministic hash (a real system uses SHA-256)
    return sum(ord(c) for c in text) % 100000
audit = {"model": "med-llm@2026-03-01", "input_hash": content_hash("visit transcript"),
         "output_hash": content_hash("SOAP note"), "clinician_signoff": True}
GATE = {                                                  # each must hold to release a medical AI output
    "PHI de-identified before the model call": True,
    "output grounded to the source (no hallucination)": True,
    "safety guardrails passed": True,
    "clinician reviewed and signed off": True,
    "audit trail written (model, hashes, timestamp)": False}   # UNMET - logging not wired up
unmet = [k for k, ok in GATE.items() if not ok]
print("Audit record: model={} input_hash={} output_hash={} signoff={}".format(
    audit["model"], audit["input_hash"], audit["output_hash"], audit["clinician_signoff"]))
print()
print("MEDICAL AI RELEASE GATE:")
for k, ok in GATE.items():
    print("  [{}] {}".format("x" if ok else " ", k))
print()
print("decision: {}".format("RELEASE" if not unmet else "BLOCK - {} unmet:".format(len(unmet))))
for u in unmet:
    print("   - " + u)
print("The audit trail is not bureaucracy: when an AI output is questioned, you must show exactly what model produced what, from")
print("what input, and which clinician signed it. De-identify, ground, guardrail, sign, and log - or it does not ship.")

# Output:
# Audit record: model=med-llm@2026-03-01 input_hash=1689 output_hash=777 signoff=True
#
# MEDICAL AI RELEASE GATE:
#   [x] PHI de-identified before the model call
#   [x] output grounded to the source (no hallucination)
#   [x] safety guardrails passed
#   [x] clinician reviewed and signed off
#   [ ] audit trail written (model, hashes, timestamp)
#
# decision: BLOCK - 1 unmet:
#    - audit trail written (model, hashes, timestamp)
# The audit trail is not bureaucracy: when an AI output is questioned, you must show exactly what model produced what, from
# what input, and which clinician signed it. De-identify, ground, guardrail, sign, and log - or it does not ship.

- Every output is logged: model version, input and output hashes, timestamp, and clinician sign-off.
- The gate checks five things: de-identified, grounded, guardrails passed, clinician signed, and audit-logged.
- Four pass but the audit trail is not wired up, so the decision is **BLOCK** — the output does not ship.
- The audit trail is not bureaucracy: it is what lets you show what model produced what, from what input, and who signed it.

#### 💡 What just happened

⚡ What just happened? The release gate is the lesson’s single decision point - de-identified, grounded, guardrailed, clinician-signed, and audit-logged, with one unmet item (the audit trail here) forcing a BLOCK - and the audit record is what makes a clinical output traceable and defensible. Tradeoff: a gate that can say no will sometimes delay a launch, which is exactly its job. That is the pattern the whole module carries: the model plus the guardrails that make it safe to ship.

- A five-item release gate reading BLOCK on the unmet audit item
- De-identified, grounded, guardrailed, signed, logged

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

## Putting It Together

> ✅ **Info**
>
> 🧠 The whole picture — the safety system around the model
> A medical AI system is not the model; it is the model plus a pipeline of guardrails, and this lesson built that pipeline as runnable checks. **Classify the use case** for the oversight it needs — assist, don’t replace (Step 1). **Validate the note structure** and catch the Assessment-without-Objective fabrication (Step 2). **Gate the Q&A before it answers** — answer, refuse-and-redirect, or escalate (Step 3). **Route on a schema field** — the critical flag that pages a radiologist (Step 4). **De-identify at the boundary** — strip PHI before any model call or log (Step 5). **Ground every claim to the source** — block the fluent fabrication (Step 6). And **gate every release** — de-identified, grounded, guardrailed, signed, and logged, or it does not ship (Step 7). Each guardrail stands on earlier work: the PHI scrub on Lesson 15.4, the schemas on Lesson 5.5, the grounding on Modules 08 and 15, the model call on Module 07. The single sentence to carry out of healthcare and into every regulated domain: the model drafts, the human decides, and the guardrails are what make the draft safe to touch a chart.

**📦 **The healthcare guardrail picker****

| Guardrail | What it protects against | What it produces | Who owns it |
|---|---|---|---|
| Oversight classifier | A model making a clinical decision | A required-control tier per use case | Clinical + product leadership |
| Structure validator | Malformed or fabricated notes/reports | A valid / block verdict + a fabrication flag | The engineering team |
| Guardrail gate | Unsafe individual advice, missed emergencies | An answer / refuse / escalate decision | Safety + clinical review |
| De-identifier | A PHI breach at the model or log boundary | De-identified text + a removed count | Privacy / security (line 2) |
| Grounding check | A hallucinated claim reaching a chart | A grounded / blocked verdict per claim | The engineering team |
| Audit + release gate | An untraceable, unsigned release | A ship / block decision + an audit record | An accountable clinical owner |

> 📦 **Info**
>
> ➡️ Where this goes nextModule 16 tours the industries where GenAI ships, and healthcare set the pattern at its highest-stakes setting: an industry AI system is the *model plus the guardrails that make it safe to ship in that industry*. The same discipline — human-in-the-loop, structure validation, guardrails, de-identification, grounding, and an audit trail — carries into the next lessons at a different intensity. The safety pattern carries forward: in Lesson 16.4 the model assists a developer, and the guardrails become the test suite, the code review, and the CI gate that keep an AI’s code changes safe; and in Lesson 16.5 the same pattern lets a PM, analyst, or marketer build with no-code tools without writing the plumbing — and shows where the guardrails still have to be. Carry one sentence forward: you do not ship a model into an industry; you ship the model wrapped in the safety system that industry requires. The reference points here are [HIPAA de-identification](https://www.hhs.gov/hipaa/for-professionals/privacy/special-topics/de-identification/index.html) and RSNA’s [structured reporting templates](https://radreport.org/). Build the safety system.

*Practice exercises are in the companion practice notebook.*

---

## 🎓 Summary

You've completed the practical part of **GenAI in Healthcare— Assist, Don’t Replace**.

### Next steps:
1. Re-run cells with different parameters to build intuition
2. Try the practice exercises (see `practice-lab/practice-lab-16_1.ipynb` if available)
3. Review the full HTML lesson for the complete narrative

*Generated from `lesson-16.1.html` - regenerate this notebook whenever the source HTML is updated.*
